In [2]:
import numpy as np
import pandas as pd

In [4]:
df=pd.read_csv("../data/archive/combine.csv",low_memory=False)

In [5]:
df.columns=df.columns.str.strip()

In [6]:
df.head()

,Destination Port,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,...,min_seg_size_forward,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
0,54865,3.0,2.0,0.0,12.0,0.0,6.0,6.0,6.0,0.0,...,20.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,BENIGN
1,55054,109.0,1.0,1.0,6.0,6.0,6.0,6.0,6.0,0.0,...,20.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,BENIGN
2,55055,52.0,1.0,1.0,6.0,6.0,6.0,6.0,6.0,0.0,...,20.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,BENIGN
3,46236,34.0,1.0,1.0,6.0,6.0,6.0,6.0,6.0,0.0,...,20.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,BENIGN
4,54863,3.0,2.0,0.0,12.0,0.0,6.0,6.0,6.0,0.0,...,20.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,BENIGN


In [7]:
df['Label']=df['Label'].str.strip().str.upper()

In [8]:
df['Label'].value_counts()

Label
BENIGN              1672837
DOS HULK             231073
PORTSCAN             158930
DDOS                 128027
DOS GOLDENEYE         10293
DOS SLOWLORIS          5796
DOS SLOWHTTPTEST       5499
BOT                    1966
INFILTRATION             36
HEARTBLEED               11
Name: count, dtype: int64

In [9]:
df['Label']=df['Label'].replace({
    'DOS HULK':'DOS',
    'DOS GOLDENEYE':'DOS',
    'DOS SLOWLORIS':'DOS',
    'DOS SLOWHTTPTEST':'DOS'})

In [10]:
df=df[df['Label'].isin(['BENIGN','DOS','DDOS','PORTSCAN'])]

In [11]:
print("Class distribution after label mapping and filtering",df['Label'].value_counts())

Class distribution after label mapping and filtering Label
BENIGN      1672837
DOS          252661
PORTSCAN     158930
DDOS         128027
Name: count, dtype: int64


In [12]:
print("Shape before cleaning",df.shape)

Shape before cleaning (2212455, 79)


In [13]:
df.replace([np.inf,-np.inf],np.nan,inplace=True)

In [14]:
df.dropna(inplace=True)

In [15]:
df.drop_duplicates(inplace=True)

In [16]:
df.reset_index(drop=True,inplace=True)

In [17]:
print("Shape after cleaning",df.shape)

Shape after cleaning (1939657, 79)


In [18]:
x=df.drop('Label',axis=1)

In [19]:
y=df['Label']

In [20]:
from sklearn.model_selection import train_test_split

x_train,x_test,y_train,y_test=train_test_split(
    x,y,
    test_size=0.3,
    random_state=42,
    stratify=y)

In [21]:
from sklearn.preprocessing import StandardScaler

scaler=StandardScaler()
x_train=scaler.fit_transform(x_train)
x_test=scaler.transform(x_test)

In [22]:
from sklearn.model_selection import train_test_split

x_train_knn, _, y_train_knn, _ = train_test_split(
    x_train,
    y_train,
    train_size=100000,
    stratify=y_train,
    random_state=42
)

In [23]:
from sklearn.neighbors import KNeighborsClassifier

model = KNeighborsClassifier(
    n_neighbors=5,
    weights='uniform',
    metric='minkowski'
)

In [24]:
model.fit(x_train_knn, y_train_knn)

,n_neighbors,5
,weights,'uniform'
,algorithm,'auto'
,leaf_size,30
,p,2
,metric,'minkowski'
,metric_params,None
,n_jobs,None


In [25]:
y_pred = model.predict(x_test)

In [26]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred, digits=6))
print(confusion_matrix(y_test, y_pred))

Accuracy: 0.9865234113195096
              precision    recall  f1-score   support

      BENIGN   0.992304  0.991184  0.991744    458161
        DDOS   0.997124  0.992996  0.995056     38405
         DOS   0.989676  0.989591  0.989634     58124
    PORTSCAN   0.870558  0.892348  0.881318     27208

    accuracy                       0.986523    581898
   macro avg   0.962416  0.966530  0.964438    581898
weighted avg   0.986667  0.986523  0.986588    581898

[[454122     63    366   3610]
 [    78  38136    191      0]
 [   558     47  57519      0]
 [  2886      0     43  24279]]


In [27]:
import joblib

joblib.dump(model, "../models/knn_model.pkl")
joblib.dump(scaler, "../models/knn_scaler.pkl")

['../models/knn_scaler.pkl']